# RAG Evaluation with LLM-as-Judge — Rubric-Based Grounding Assessment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/evaluation_grouse.ipynb)

Automated metrics like BLEU or F1 capture surface-level similarity but miss **nuance** — is the answer *grounded* in the retrieved context? Is it *complete*? Is it *coherent*? This notebook implements a rubric-based LLM-as-judge evaluation framework inspired by the Grouse (Grounded Scoring Engine) methodology.

Where valuation_deep_eval.ipynb gives you numbers, this notebook gives you **structured reasoning** about *why* an answer is good or bad.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **when automated metrics fall short** and why LLM-as-judge matters
- Design **multi-dimensional evaluation rubrics** for RAG systems
- Implement a **structured judge pipeline** using an LLM to score groundedness, completeness, relevance, and coherence
- Analyze **judge reliability** through inter-rater agreement and calibration
- Compare **metric-based vs rubric-based evaluation** on the same RAG outputs

## Part 1 — Why Metrics Aren't Enough

Consider two RAG answers to "What causes aurora borealis?":

**Answer A**: "Aurora borealis is caused by charged particles from the sun interacting with Earth's magnetic field and atmosphere."
**Answer B**: "The northern lights are a beautiful natural light display. They can be seen near the Arctic. Many tourists travel to see them."

A simple token-overlap metric might score both similarly (both mention the topic), but any human sees that **A is grounded and correct** while **B is fluent but empty** — it never explains the cause.

This is the gap that LLM-as-judge evaluation fills: it applies **human-like reasoning** through structured rubrics, producing not just a score but an **explanation** of what's right and wrong.

### The Evaluation Dimensions

Rubric-based RAG evaluation typically scores along four axes:

| Dimension | Question It Answers | Scale |
|-----------|-------------------|-------|
| **Groundedness** | Is every claim supported by the retrieved context? | 1-5 |
| **Completeness** | Does the answer cover all relevant information from the context? | 1-5 |
| **Relevance** | Does the answer address the user's actual question? | 1-5 |
| **Coherence** | Is the answer well-organized, fluent, and non-contradictory? | 1-5 |

Each dimension captures a different failure mode:
- Low groundedness → hallucination (answer invents facts not in context)
- Low completeness → omission (answer ignores relevant retrieved info)
- Low relevance → drift (answer is correct but doesn't address the question)
- Low coherence → garbled (answer has contradictions or poor structure)

## Part 2 — Environment Setup

In [ ]:
# ============================================================
# Cell 1: Install dependencies
# ============================================================
!pip install -q transformers torch bitsandbytes accelerate sentence-transformers faiss-cpu datasets
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# Cell 2: Load the judge model (Qwen3-8B, 4-bit)
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto"
)
print(f"Judge model loaded: {model_name}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Part 3 — Designing Evaluation Rubrics

The power of LLM-as-judge comes from **well-designed rubrics**. A rubric tells the judge model exactly what to look for and how to score it. Vague rubrics produce inconsistent scores; precise rubrics produce reliable ones.

### Rubric Design Principles

1. **Anchored scales**: Each score level (1-5) has a concrete definition, not just "bad" to "good"
2. **Observable criteria**: Score based on what's *in* the text, not what the judge *thinks*
3. **One dimension per rubric**: Don't mix groundedness and coherence in one prompt
4. **Examples**: Include a scored example so the judge calibrates

In [ ]:
# ============================================================
# Cell 3: Define evaluation rubrics
# ============================================================

RUBRICS = {
    "groundedness": {
        "description": "Is every claim in the answer supported by the retrieved context?",
        "scale": {
            1: "Most claims are fabricated — not present in context at all",
            2: "Several unsupported claims mixed with some supported ones",
            3: "Core claims are supported but some details are added or inferred",
            4: "Nearly all claims are grounded; minor inferences are reasonable",
            5: "Every claim is directly traceable to the retrieved context"
        }
    },
    "completeness": {
        "description": "Does the answer cover all relevant information available in the context?",
        "scale": {
            1: "Misses most relevant information from the context",
            2: "Covers only one aspect; ignores other relevant parts",
            3: "Covers the main point but misses important supporting details",
            4: "Covers most relevant information with minor omissions",
            5: "Comprehensively covers all relevant information from context"
        }
    },
    "relevance": {
        "description": "Does the answer directly address the user's question?",
        "scale": {
            1: "Completely off-topic or addresses a different question",
            2: "Tangentially related but doesn't answer the actual question",
            3: "Addresses the question partially or with unnecessary tangents",
            4: "Directly addresses the question with minor tangential content",
            5: "Precisely and completely addresses exactly what was asked"
        }
    },
    "coherence": {
        "description": "Is the answer well-structured, fluent, and internally consistent?",
        "scale": {
            1: "Incoherent, contradictory, or incomprehensible",
            2: "Understandable but poorly organized with some contradictions",
            3: "Adequate structure but could be clearer or more concise",
            4: "Well-organized and clear with good flow",
            5: "Excellently structured, concise, and reads naturally"
        }
    }
}

print(f"Defined {len(RUBRICS)} evaluation dimensions")
for dim, rubric in RUBRICS.items():
    print(f"  {dim}: {rubric['description']}")

In [ ]:
# ============================================================
# Cell 4: Build the judge prompt template
# ============================================================

def build_judge_prompt(question, context, answer, dimension, rubric):
    scale_text = "\n".join(f"  {k}: {v}" for k, v in rubric["scale"].items())
    
    prompt = f"""You are an expert evaluator assessing the quality of a RAG (Retrieval-Augmented Generation) system's answer.

## Task
Evaluate the following answer on the dimension of **{dimension}**.

## Definition
{rubric["description"]}

## Scoring Scale
{scale_text}

## Inputs

**User Question:** {question}

**Retrieved Context:**
{context}

**Generated Answer:**
{answer}

## Instructions
1. Analyze the answer against the rubric criteria
2. Provide a brief explanation (2-3 sentences) justifying your score
3. Output your response in this exact format:

EXPLANATION: [your reasoning]
SCORE: [1-5]"""
    return prompt

# Test the prompt
sample_prompt = build_judge_prompt(
    question="What is photosynthesis?",
    context="Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose and oxygen.",
    answer="Photosynthesis converts sunlight into food for plants.",
    dimension="completeness",
    rubric=RUBRICS["completeness"]
)
print("Judge prompt length:", len(sample_prompt), "chars")
print("First 200 chars:", sample_prompt[:200])

In [ ]:
# ============================================================
# Cell 5: Implement the judge pipeline
# ============================================================
import re

def judge_answer(question, context, answer, dimension, rubric):
    prompt = build_judge_prompt(question, context, answer, dimension, rubric)
    
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=256, temperature=0.1,
            do_sample=True, top_p=0.95
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Parse score and explanation
    score_match = re.search(r'SCORE:\s*(\d)', response)
    expl_match = re.search(r'EXPLANATION:\s*(.+?)(?=SCORE:|$)', response, re.DOTALL)
    
    score = int(score_match.group(1)) if score_match else None
    explanation = expl_match.group(1).strip() if expl_match else response.strip()
    
    return {"dimension": dimension, "score": score, "explanation": explanation, "raw": response}

def evaluate_rag_output(question, context, answer):
    results = {}
    for dim, rubric in RUBRICS.items():
        result = judge_answer(question, context, answer, dim, rubric)
        results[dim] = result
        print(f"  {dim}: {result['score']}/5 — {result['explanation'][:80]}...")
    
    avg = sum(r['score'] for r in results.values() if r['score']) / len(results)
    print(f"  AVERAGE: {avg:.1f}/5")
    return results

print("Judge pipeline ready!")

## Part 4 — Running the Evaluation

Let's build a small RAG system and evaluate its outputs. We'll create test cases with known-good context and questions, then judge the generated answers.

In [ ]:
# ============================================================
# Cell 6: Create test cases and run evaluation
# ============================================================

test_cases = [
    {
        "question": "How does a transformer model process input text?",
        "context": "A transformer model processes input text by first tokenizing it into subword tokens. Each token is converted to an embedding vector. Positional encodings are added to preserve order information. The embeddings then pass through multiple layers of self-attention and feed-forward networks. Self-attention allows each token to attend to all other tokens, capturing contextual relationships.",
        "answer": "A transformer processes text by tokenizing it into subwords, converting to embeddings with positional encodings, then passing through self-attention and feed-forward layers. Self-attention lets each token attend to all others for context."
    },
    {
        "question": "What are the benefits of LoRA for finetuning?",
        "context": "LoRA (Low-Rank Adaptation) reduces finetuning cost by freezing the original model weights and training small low-rank matrices. This reduces trainable parameters by 10,000x while maintaining performance. LoRA adapters are small (often under 100MB) and can be swapped at serving time.",
        "answer": "LoRA is great for finetuning because it makes models much better at everything. It was invented in 2024 and is used by all major AI companies. The technique works by changing the model's architecture completely."
    },
]

print("=" * 60)
for i, tc in enumerate(test_cases):
    print(f"\nTest Case {i+1}: {tc['question'][:50]}...")
    print("-" * 40)
    results = evaluate_rag_output(tc['question'], tc['context'], tc['answer'])
    print()

## Part 5 — Judge Reliability Analysis

A critical question: **can we trust the judge?** If the same answer gets score 3 on one run and score 5 on another, the evaluation is useless. Let's measure consistency.

### Inter-Run Agreement
We run the judge multiple times on the same input and measure score variance. Low temperature (0.1) should produce consistent scores, but let's verify.

In [ ]:
# ============================================================
# Cell 7: Measure judge consistency
# ============================================================

tc = test_cases[0]
dimension = "groundedness"
rubric = RUBRICS[dimension]
n_runs = 3

scores = []
for run in range(n_runs):
    result = judge_answer(tc['question'], tc['context'], tc['answer'], dimension, rubric)
    scores.append(result['score'])
    print(f"  Run {run+1}: {result['score']}/5")

if all(s is not None for s in scores):
    import statistics
    mean = statistics.mean(scores)
    stdev = statistics.stdev(scores) if len(scores) > 1 else 0
    print(f"\nMean: {mean:.2f}, Std Dev: {stdev:.2f}")
    print(f"Consistency: {'HIGH' if stdev < 0.5 else 'MEDIUM' if stdev < 1.0 else 'LOW'}")
else:
    print("Some runs failed to produce valid scores")

## Part 6 — Comparing Metrics vs Rubric-Based Evaluation

Let's run both approaches on the same outputs and see where they agree and disagree. This highlights the unique value of each approach.

Key insight: **Use metrics for fast automated sweeps, use rubric-based judges for deep analysis of failure cases.** They complement each other — metrics tell you *that* something is wrong, rubrics tell you *what* is wrong.

In [ ]:
# ============================================================
# Cell 8: Side-by-side comparison
# ============================================================

from collections import Counter

# Simple metric: token overlap (proxy for ROUGE-like metrics)
def token_overlap(answer, context):
    a_tokens = set(answer.lower().split())
    c_tokens = set(context.lower().split())
    if not a_tokens:
        return 0.0
    precision = len(a_tokens & c_tokens) / len(a_tokens)
    recall = len(a_tokens & c_tokens) / len(c_tokens) if c_tokens else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return f1

print("Metric vs Rubric Comparison")
print("=" * 60)
for i, tc in enumerate(test_cases):
    overlap = token_overlap(tc['answer'], tc['context'])
    judge_result = judge_answer(tc['question'], tc['context'], tc['answer'], 'groundedness', RUBRICS['groundedness'])
    
    print(f"\nCase {i+1}: {tc['question'][:50]}...")
    print(f"  Token overlap F1: {overlap:.3f}")
    print(f"  Judge groundedness: {judge_result['score']}/5")
    print(f"  Judge says: {judge_result['explanation'][:100]}...")

## 🏋️ Exercises

**Exercise 1 — Custom Rubric:** Design a rubric for "Conciseness" (does the answer avoid unnecessary verbosity?). Implement it and test on the sample cases. Does it correlate with or contradict the coherence dimension?

**Exercise 2 — Adversarial Cases:** Create 3 test cases specifically designed to fool the judge: one with fluent hallucinations, one with correct but incoherent text, one with grounded but irrelevant content. How does the judge handle each?

**Exercise 3 — Circular Evaluation:** This notebook uses the same model (Qwen3-8B) as both generator and judge. Test what happens when you use a smaller model as generator and the 8B as judge. Does separation improve reliability?

## 📝 Key Takeaways

- **Automated metrics miss nuance** — token overlap can't detect hallucination or completeness gaps
- **Rubric design is the bottleneck** — well-anchored scales with clear criteria produce reliable judges
- **LLM judges need calibration** — run multiple times, check consistency before trusting scores
- **Four dimensions capture most RAG failures**: groundedness, completeness, relevance, coherence
- **Metrics and rubrics are complementary** — use metrics for sweeps, rubrics for diagnosis
- **Circular evaluation is a known limitation** — using the same model as generator and judge requires caution

## 📚 References & Further Reading

- [RAGAS: Automated Evaluation of RAG](https://arxiv.org/abs/2309.15217) — foundational RAG evaluation framework
- [Judging LLM-as-a-Judge](https://arxiv.org/abs/2306.05685) — MT-Bench and systematic analysis of LLM judges
- [G-Eval: NLG Evaluation using GPT-4](https://arxiv.org/abs/2303.16634) — chain-of-thought evaluation
- Related notebooks: [evaluation_deep_eval.ipynb](evaluation_deep_eval.ipynb), [evals/06_rubrics_and_llm_as_judge.ipynb](../evals/06_rubrics_and_llm_as_judge.ipynb)